### This notebook demonstrate how to use the PlateModelManager to access plate model files.

In [ ]:
import os, warnings
from pathlib import Path
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from plate_model_manager import PlateModelManager, PresentDayRasterManager
from gplately import PlateReconstruction, PlotTopologies
from gplately.commands.list_models import get_model_names

warnings.filterwarnings("ignore", category=UserWarning, module="gplately")

pm_manager = PlateModelManager()

#### Get the names of all available models in the PlateModelManager

In [ ]:
gplately_model_names = get_model_names()
for name in pm_manager.get_available_model_names():
    # the pm_manager.get_available_model_names() returns a superset of GPlately models.
    # we need to check the the model name agaist a list of GPlately officially supported models.
    # https://gplates.github.io/gplately/latest/sphinx/html/plate_models.html
    if name in gplately_model_names:
        print(name)

#### Download model "Muller2019" and put the files in folder "plate-model-repo"

In [ ]:
model = pm_manager.get_model("Muller2019")
assert model
model.set_data_dir("plate-model-repo")
for layer in model.get_avail_layers():
    model.get_layer(layer)

# now let's see what are inside the "plate-model-repo/muller2019" folder
print(os.listdir("plate-model-repo/muller2019"))

#### List all vailable layers in model Muller2019

In [ ]:
for layer in model.get_avail_layers():
    print(layer)

#### Download rotation files

In [ ]:
rotation_files = model.get_rotation_model()
print(rotation_files)

#### Download static polygons

In [ ]:
static_polygon_files = model.get_layer("StaticPolygons")
print(static_polygon_files)

#### Download Coastlines

In [ ]:
coasts_files = model.get_layer("Coastlines")
print(coasts_files)

#### Download all layers

In [ ]:
for layer in model.get_avail_layers():
    print(model.get_layer(layer))

#### Get a list of time dependent rasters

In [ ]:
for raster in model.get_avail_time_dependent_raster_names():
    print(raster)

#### Download AgeGrids rasters

In [ ]:
import warnings

warnings.filterwarnings("ignore")
print(model.get_rasters("AgeGrids", times=[10, 20, 30]))
print(model.get_raster("AgeGrids", time=100))

#### Download AgeGrids rasters for all available times

This function will take a while to finish and download a large volume data. Uncomment the code in the code cell below to try it.

In [ ]:
# model.download_time_dependent_rasters("AgeGrids")

#### List the names of all present-day rasters

In [ ]:
print(PresentDayRasterManager().list_present_day_rasters())

#### Get "topography" present-day raster

In [ ]:
print(PresentDayRasterManager().get_raster("topography"))

In [ ]:
# use `PlateModelManager` to create `PlateReconstruction` and `PlotTopologies` objects
model = PlateModelManager().get_model("Zahirovic2022")
model.set_data_dir("plate-model-repo")  # type: ignore

age = 55
test_model = PlateReconstruction(
    model.get_rotation_model(),  # type: ignore
    topology_features=model.get_layer("Topologies"),  # type: ignore
    static_polygons=model.get_layer("StaticPolygons"),  # type: ignore
)
gplot = PlotTopologies(
    test_model,
    coastlines=model.get_layer("Coastlines"),  # type: ignore
    COBs=model.get_layer("COBs"),  # type: ignore
    time=age,
)

fig = plt.figure(figsize=(12, 6), dpi=72)
ax = fig.add_subplot(111, projection=ccrs.Robinson(central_longitude=180))
ax.set_global()  # type: ignore

# now use PlotTopologies object to plot some model data
gplot.plot_continent_ocean_boundaries(ax, color="cornflowerblue")
gplot.plot_coastlines(ax, color="black")
gplot.plot_all_topological_sections(
    ax,
    plot_subduction_teeth=True,
    other_kwargs={
        "color": "grey",
        "linewidth": 0.5,
    },
    ridge_kwargs={
        "color": "black",
        "linewidth": 0.7,
    },
    transform_kwargs={
        "color": "green",
        "linewidth": 0.7,
    },
    trench_kwargs={
        "color": "blue",
        "linewidth": 0.7,
    },
)

plt.title(f"{age} Ma")

# save the map as a .png file
data_dir = Path("./gplately-example-data")
data_dir.mkdir(parents=True, exist_ok=True)
output_file = data_dir / "02-PlateModelManager.png"
fig.savefig(output_file, dpi=120, bbox_inches="tight")  # transparent=True)
print(f"Done! The output file {output_file} has been saved.")

plt.show()
plt.close(fig)